# 06 — Feature Engineering

**Objetivo:** preparar a base modelável com transformações **aprendidas somente no treino** e aplicadas sem reajuste na validação e no teste.

**Regras anti-leakage:**
1. Winsorização: limites P1/P99 calculados **no treino**
2. WOE do `pagamento_predominante`: mapeamento aprendido **no treino**
3. Log-transformação de valores monetários (monotônica — sem risco de vazamento)
4. Standardização: feita no `08_modelagem` dentro do `Pipeline` (fit no treino)
5. Split por **sorteio de clientes** (75% treino / 25% validação) **no mesmo período** (06–12/05) — nunca aleatório por linha; teste out-of-time nas safras 13–15/05

**Outputs:** `train_base`, `validation_base`, `test_base` (o teste carrega a coluna `safra` — 1 safra por dia: 13, 14 e 15/05)

In [0]:
# ── Configuração central (idem notebook 00) ──
CATALOG = "workspace"
SCHEMA  = "ml_bakehouse"

# Split (validado no notebook 00):
#  - Treino e validação NO MESMO PERÍODO (06–12/05), por sorteio de clientes 75/25
#  - Teste out-of-time: 3 safras de 1 dia (13, 14 e 15/05), todos os clientes
SPLIT_IN_SAMPLE = ("2024-05-06", "2024-05-12")   # treino + validação (mesmo período)
SPLIT_OOT       = ("2024-05-13", "2024-05-15")   # 3 safras de 1 dia
VAL_FRACTION    = 0.25                          # fração de clientes reservada à validação
SPLIT_SEED      = 42                            # sorteio determinístico (reprodutível)

import numpy as np
import pandas as pd
from pyspark.sql import functions as F

features  = spark.table(f"{CATALOG}.{SCHEMA}.customer_features")
target_tb = spark.table(f"{CATALOG}.{SCHEMA}.population_target")
df_model  = (
    features
    .join(target_tb, on=["reference_date", "customerID"], how="inner")
)

in_sample = df_model.filter(F.col("reference_date").between(*SPLIT_IN_SAMPLE))
test_df   = (df_model.filter(F.col("reference_date").between(*SPLIT_OOT))
             .withColumn("safra", F.date_format("reference_date", "yyyyMMdd")))

# Sorteio determinístico de clientes: 75% treino / 25% validação (mesmo período)
val_clientes = (in_sample.select("customerID").distinct()
                .sample(withReplacement=False, fraction=VAL_FRACTION, seed=SPLIT_SEED)
                .toPandas()["customerID"].tolist())

train_df = in_sample.filter(~F.col("customerID").isin(val_clientes))
val_df   = in_sample.filter(F.col("customerID").isin(val_clientes))

print("Treino:", train_df.count(), "| Validação:", val_df.count(), "| Teste OOT (3 safras):", test_df.count())
n_clientes_tr = train_df.select("customerID").distinct().count()
print(f"Clientes — treino: {n_clientes_tr} | validação: {len(val_clientes)} | interseção: {train_df.select('customerID').distinct().filter(F.col('customerID').isin(val_clientes)).count()}")
print("Taxa evento — treino:", round(train_df.agg(F.avg("target")).collect()[0][0], 4),
      "| validação:", round(val_df.agg(F.avg("target")).collect()[0][0], 4))

In [0]:
# 1) Winsorização — limites P1/P99 aprendidos no TREINO
feat_num = [
    "recencia_dias", "freq_compras_5d", "valor_total_5d", "ticket_medio_5d",
    "qtde_itens_5d", "produtos_distintos_5d", "franquias_distintas_5d",
    "preco_medio_5d", "tendencia_2d_vs_3a5",
]

bounds = {c: train_df.approxQuantile(c, [0.01, 0.99], 0.001) for c in feat_num}

def winsorizar(df):
    for c in feat_num:
        lo, hi = bounds[c]
        df = df.withColumn(c, F.least(F.greatest(F.col(c), F.lit(lo)), F.lit(hi)))
    return df

train_w = winsorizar(train_df)
val_w   = winsorizar(val_df)      # mesmos limites do treino — sem reajuste
test_w  = winsorizar(test_df)

# 2) Log nas variáveis monetárias (assimetria à direita)
def log_monetarias(df):
    return (df
        .withColumn("valor_total_log", F.log1p("valor_total_5d")).drop("valor_total_5d")
        .withColumn("ticket_medio_log", F.log1p("ticket_medio_5d")).drop("ticket_medio_5d"))

train_w = log_monetarias(train_w)
val_w   = log_monetarias(val_w)
test_w  = log_monetarias(test_w)

feat_final = [c for c in train_w.columns if c not in ("customerID", "reference_date", "target", "primeira_compra_janela", "ultima_compra_janela", "compras_historia", "pagamento_predominante")]
print("Features numéricas finais:", feat_final)

In [0]:
# 3) WOE do pagamento_predominante — mapeamento aprendido NO TREINO
def mapa_woe_categorica(train_pdf, col, target="target"):
    g = train_pdf.groupby(col)[target].agg(["sum", "count"])
    g["nao_eventos"] = g["count"] - g["sum"]
    g["pct_ev"]  = g["sum"] / g["sum"].sum()
    g["pct_nev"] = g["nao_eventos"] / g["nao_eventos"].sum()
    g["woe"] = np.log((g["pct_ev"] + 1e-6) / (g["pct_nev"] + 1e-6))
    return g["woe"].to_dict()

train_pd = train_w.select("pagamento_predominante", "target").toPandas()
woe_map = mapa_woe_categorica(train_pd, "pagamento_predominante")
print("WOE por forma de pagamento:", {k: round(v, 4) for k, v in woe_map.items()})

def aplicar_woe(df):
    # create_map: lookup do WOE por categoria; coalesce → WOE = 0 para categoria desconhecida (não vista no treino)
    mapeamento = F.create_map([F.lit(x) for kv in woe_map.items() for x in kv])
    return (df.withColumn("pagamento_woe",
                F.coalesce(mapeamento[F.col("pagamento_predominante")], F.lit(0.0)).cast("double"))
            .drop("pagamento_predominante"))

train_w = aplicar_woe(train_w)
val_w   = aplicar_woe(val_w)   # mesmo mapeamento — sem reajuste
test_w  = aplicar_woe(test_w)

display(train_w.groupBy(F.round("pagamento_woe", 3)).agg(F.count("*").alias("linhas")).orderBy(F.desc("linhas")))

In [0]:
# 4) Gravação das bases modeláveis
for df_, nome in [(train_w, "train_base"), (val_w, "validation_base"), (test_w, "test_base")]:
    colunas = ["customerID", "reference_date"] + (["safra"] if "safra" in df_.columns else []) + ["target", *feat_final, "pagamento_woe"]
    (df_.select(*colunas)
       .write.mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(f"{CATALOG}.{SCHEMA}.{nome}"))
    print(f"Salvo: {CATALOG}.{SCHEMA}.{nome}")

display(spark.table(f"{CATALOG}.{SCHEMA}.train_base").limit(10))

## Verificações anti-leakage

* ✅ Winsorização e WOE usam apenas estatísticas do **treino** (funções `winsorizar`/`aplicar_woe` recebem os limites/mapas calculados em `train_df`)
* ✅ Split por cliente: validação usa 25% dos clientes sorteados **sem interseção** com o treino (mesmo período 06–12/05); teste OOT é **posterior** (13–15/05, 3 safras com `safra`)
* ✅ Nenhuma variável usa `tx_date > reference_date - 1` (janela futura é só do target, construído no `03_target`)
* ⚠️ Para re-executar este notebook após mudanças, rode 01→02→03 primeiro

**Próximo passo:** `07_sampling` — verificar se há desbalanceamento relevante.